# Chapter 30 — Treat Prompts as Programs

**Book alignment:** Debugging AI From First Principles, Chapter 30

**Question this notebook isolates:** An engineer "tightens" the system prompt in the
dashboard and refund answers start citing reference numbers that do not exist. Does the
prompt-as-program discipline — repo, hash, byte-diff, re-assemble, fixture-gate — separate
**H1** (prompt regression, environment fixed) from **H2** (environment drift, prompt hash
fixed) from **H3** (no fixture suite ever existed, so "regression" is unmeasurable)?

In [ ]:
import hashlib
def ph(text): return "sys@" + hashlib.sha1(text.encode()).hexdigest()[:4]

V14 = ("You are a refund assistant.\n"
       "GUARDRAIL: cite only reference numbers present in the retrieved context.\n")
V15 = ("You are a refund assistant.\n"
       "Be concise.\n")                                  # the 'tightening': + concise, - guardrail

INDEX_SNAPSHOT = "idx-2026-08-14"                          # environment pin, held fixed

## 1. Hash and byte-diff the two versions

In [ ]:
import difflib
print("v14 hash:", ph(V14))
print("v15 hash:", ph(V15))
assert ph(V14) != ph(V15)
diff = list(difflib.unified_diff(V14.splitlines(), V15.splitlines(), "v14", "v15", lineterm=""))
for line in diff:
    print(line)
guardrail_touched = any(l.startswith("-") and "GUARDRAIL" in l for l in diff)
assert guardrail_touched
print("\nthe diff shows it: + 'Be concise'  AND  - the GUARDRAIL line (the real break)")

## 2. Re-assemble the failing input under each version; run the pinned fixture suite

In [ ]:
FIXTURES = [
    ("refund status?",  "cite only present refs",  "RB-1 is in context"),
    ("split shipment?", "cite only present refs",  "4.2-exception is in context"),
    ("older refund?",   "cite only present refs",  "nothing relevant in context"),
]

def answer(prompt_text, fixture):
    q, _, ctx = fixture
    has_guardrail = "GUARDRAIL" in prompt_text
    if "nothing relevant" in ctx and not has_guardrail:
        return "your refund reference is RB-8814"           # fabricated when the guardrail is gone
    return "citing only refs present in context"

v14_pass = sum(1 for fx in FIXTURES if "RB-8814" not in answer(V14, fx))
v15_pass = sum(1 for fx in FIXTURES if "RB-8814" not in answer(V15, fx))
print(f"v14 (idx {INDEX_SNAPSHOT}): {v14_pass}/3 fixtures pass")
print(f"v15 (idx {INDEX_SNAPSHOT}): {v15_pass}/3 fixtures pass   <- fabricated-citation fixture red")
assert v14_pass == 3 and v15_pass < 3
print("\nfailure follows the PROMPT hash with the index snapshot fixed -> H1 (prompt regression)")

## 3. The deploy-from-hash gate: a red suite blocks the ship

In [ ]:
def deploy(prompt_text):
    passed = sum(1 for fx in FIXTURES if "RB-8814" not in answer(prompt_text, fx))
    if passed < len(FIXTURES):
        return f"BLOCKED: {passed}/{len(FIXTURES)} fixtures - REVERT, no override path"
    return f"DEPLOYED: {ph(prompt_text)} ({passed}/{len(FIXTURES)} green)"

print("v15:", deploy(V15))
print("v16 (guardrail restored):", deploy(V14))          # v16 == v14 text
assert deploy(V15).startswith("BLOCKED")
assert deploy(V14).startswith("DEPLOYED")
print('\na red suite blocks the ship even when the prose "reads better" - taste never overrides count')

## What we earned

A prompt that lives only in a dashboard textbox is a rumor about an asset. Repo-ing both
versions, hashing them, and running a byte-diff revealed the "tightening" for what it was:
`+ "Be concise"` **and** a silently deleted `GUARDRAIL` line. The pinned fixture suite —
run with the index snapshot held fixed — passed 3/3 on v14 and failed the fabricated-
citation fixture on v15, convicting **H1 (prompt regression)**. The deploy-from-hash gate
then blocks any hash that fails the suite, no override path.

**Notebook 31 / Chapter 31** applies removal-until-break to the prompt's *contents*: which
words inside it are load-bearing.